# picoRSI quickstart

A small VLM improves itself on verifiable multimodal tasks: it acts, a verifier scores what it did,
GRPO (written from scratch) turns that into a LoRA update, and a gate keeps the update only if it
helps on a separate gate split. Held-out tasks are never used to decide anything.

Runtime: **GPU** (Runtime → Change runtime type → T4 or better) for the real run; the first smoke
cell also works on CPU.

In [ ]:
REPO = "https://github.com/patrickamadeus/picorsi"   # set to the repository you cloned this notebook from
!git clone -q $REPO picorsi
%cd picorsi
!pip -q install -e ".[plot]"

## 1. Close a loop on CPU with a tiny random policy (~1 min)
This checks the plumbing, not learning: a randomly initialized model can't solve anything.

In [ ]:
!python train.py --config tiny --run_dir runs/smoke

## 2. GRPO on held-out counting

SmolVLM-256M counts blue squares in procedurally generated images. The self-curriculum keeps it
practising tasks at its competence frontier; every update has to clear the gate.

In [ ]:
!python train.py --config count_grpo --iters 30 --eval_every 5 --run_dir runs/colab

## 3. Success against cost
Always plot against cost, not iterations: update modes differ wildly in how much compute an update costs.

In [ ]:
!python -m harness.plot runs/colab --x gpu_seconds --out colab.png
from IPython.display import Image
Image("colab.png")

## 4. What happened, update by update

In [ ]:
from harness.log import Trace
tr = Trace("runs/colab")
for u in tr.records("update"):
    gb, ga = u.get("gate_before") or [None, None], u.get("gate_after") or [None, None]
    print(u["iter"], "committed" if u["committed"] else "reverted ", u.get("reason") or "",
          "| gate success", gb[0], "->", ga[0])

Next: `train.py` is the whole loop in one file, and `docs/architecture.md` shows how to add your own environment, judge or update mode.